<div style="display:flex; align-items:flex-start; gap:2rem; padding:1rem 0;">

  <div style="flex-shrink:0;">
    <img src="https://cdn.oreillystatic.com/images/sitewide-headers/oreilly_logo_mark_red.svg"
         style="height:36px; display:block; margin-bottom:0.75rem;"/>
    <img src="https://i.imgur.com/ITL8dZE.jpeg"
         style="width:260px; border-radius:4px; box-shadow:0 2px 8px rgba(0,0,0,0.15); display:block;"/>
  </div>
  <div style="flex:1; font-family:sans-serif;">
    <h1 style="font-size:1.6rem; font-weight:600; margin:0 0 0.25rem 0;">
      AI, ML and GenAI in the Lakehouse
    </h1>


   Name:          chapter 04-05-Model Deployment

   Author:    Bennie Haelen
   Date:      10-13-2024

   Purpose:   This notebook demonstrates how to govern and deploy the registered
              hotel cancellation classifier to a Databricks Model Serving endpoint.

      An outline of the different sections in this notebook:
        1 - Verify the registered model in Unity Catalog
              1-1 Inspect all registered versions
              1-2 Review model metadata and lineage
        2 - Assign the Champion alias
              2-1 Understand the Champion/Challenger pattern
              2-2 Assign the alias to the best model version
              2-3 Verify loading by alias
        3 - Deploy to a Model Serving endpoint
              3-1 Create the serving endpoint
              3-2 Wait for the endpoint to become ready
              3-3 Query the endpoint with sample data
        4 - Update the endpoint with a new model version

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

import requests
import json
import pandas as pd
import numpy as np
import time

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## Unity Catalog coordinates

The model was registered to Unity Catalog by the model building notebook.
All cells in this notebook reference the same catalog, schema, and model name.
Update these values if your setup uses different names.

In [0]:
catalog    = "book_ai_ml_lakehouse"
schema     = "default"
model_name = "hotel_cancellation_classifier"

model_uc_path = f"{catalog}.{schema}.{model_name}"
print(f"Model registry path: {model_uc_path}")

client = MlflowClient()

## Verify the registered model in Unity Catalog

Before deploying, confirm that the model building notebook successfully registered
the model and inspect the available versions. Each time `mlflow.register_model()`
is called with the same model name, MLflow increments the version number automatically.

In [0]:
# Retrieve all registered versions for this model
versions = client.search_model_versions(f"name='{model_uc_path}'")

if not versions:
    raise ValueError(
        f"No versions found for {model_uc_path}. "
        "Run the model building notebook first to register the model."
    )

print(f"Found {len(versions)} version(s) for {model_uc_path}:\n")
for v in sorted(versions, key=lambda x: int(x.version)):
    print(f"  Version {v.version}")
    print(f"    Source run:  {v.run_id}")
    print(f"    Status:      {v.status}")
    print(f"    Created:     {v.creation_timestamp}")
    print()

### Review model metadata and lineage

Unity Catalog records the lineage between the registered model and the MLflow run
that produced it. The run in turn links back to the feature table used for training,
giving us an end-to-end audit trail from raw data through to the deployed artifact.

In [0]:
# Retrieve the registered model metadata
registered_model = client.get_registered_model(model_uc_path)
print(f"Name:        {registered_model.name}")
print(f"Description: {registered_model.description or '(none)'}")
print(f"Tags:        {registered_model.tags}")

# Get the latest version details
latest = max(versions, key=lambda v: int(v.version))
print(f"\nLatest version: {latest.version}")
print(f"Source run ID:  {latest.run_id}")

# Pull the run to show training metrics
run = client.get_run(latest.run_id)
metrics = run.data.metrics
print(f"\nTraining metrics from source run:")
for k, v in sorted(metrics.items()):
    print(f"  {k}: {v:.4f}")

## Assign the Champion alias

Unity Catalog uses aliases rather than stage transitions (Staging, Production) to
govern model promotion. An alias is a named pointer that can be reassigned to any
version, making it possible to change which version is active in production without
modifying the code that loads the model.

The most common pattern in enterprise deployments is Champion/Challenger:

- **Champion**: the current production model, serving live traffic
- **Challenger**: a candidate model being evaluated in parallel

When a Challenger demonstrates better performance, it is promoted to Champion by
reassigning the alias, and the previous Champion is either archived or becomes
the new Challenger. This enables safe, zero-downtime model updates.

In [0]:
# Get the latest version number dynamically
latest_version = max(versions, key=lambda v: int(v.version)).version
print(f"Assigning Champion alias to version {latest_version}")

# Assign the Champion alias to the latest version
client.set_registered_model_alias(
    name=model_uc_path,
    alias="Champion",
    version=latest_version
)

print(f"Champion alias set: {model_uc_path}@Champion -> version {latest_version}")

### Verify loading by alias

Once the alias is assigned, any code that loads the model can use the alias URI
instead of a specific version number. This is the recommended pattern for production
code because it automatically points to whichever version holds the alias:

In [0]:
# Load the model using the Champion alias
alias_uri = f"models:/{model_uc_path}@Champion"
champion_model = mlflow.sklearn.load_model(alias_uri)

print(f"Loaded model via alias: {alias_uri}")
print(f"Model type: {type(champion_model).__name__}")

## Deploy to a Model Serving endpoint

Databricks Model Serving exposes the registered model as a scalable REST endpoint.
Once deployed, any application can call the endpoint with a JSON payload and receive
predictions in real time, without needing access to the Databricks workspace or any
Python dependencies.

We use the Databricks REST API to create the endpoint programmatically. You will need
a personal access token with the `CAN_MANAGE` permission on the serving endpoints.
Store it as a Databricks secret rather than hardcoding it into the notebook.

In [0]:
# Retrieve workspace URL and token from Databricks context
# In a Databricks notebook these are available from the notebook context
workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
token         = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

endpoint_name  = "hotel-cancellation-classifier"
served_version = latest_version   # the version we just aliased as Champion

print(f"Workspace: {workspace_url}")
print(f"Endpoint name: {endpoint_name}")
print(f"Serving model version: {served_version}")

### Create the serving endpoint

In [0]:
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

endpoint_config = {
    "name": endpoint_name,
    "config": {
        "served_models": [
            {
                "model_name":    model_uc_path,
                "model_version": str(served_version),
                "workload_size": "Small",
                "scale_to_zero_enabled": True
            }
        ]
    }
}

# Check if the endpoint already exists
list_url = f"{workspace_url}/api/2.0/serving-endpoints"
existing  = requests.get(list_url, headers=headers).json()
endpoint_names = [e["name"] for e in existing.get("endpoints", [])]

if endpoint_name in endpoint_names:
    print(f"Endpoint '{endpoint_name}' already exists. Skipping creation.")
else:
    response = requests.post(list_url, headers=headers, json=endpoint_config)
    if response.status_code == 200:
        print(f"Endpoint '{endpoint_name}' creation initiated successfully.")
    else:
        print(f"Error creating endpoint: {response.status_code}")
        print(response.json())

### Wait for the endpoint to become ready

Endpoint creation typically takes two to five minutes. The cell below polls
the endpoint state every 30 seconds and proceeds once the endpoint is ready.

In [0]:
def wait_for_endpoint(workspace_url, headers, endpoint_name, timeout_minutes=10):
    status_url   = f"{workspace_url}/api/2.0/serving-endpoints/{endpoint_name}"
    deadline     = time.time() + timeout_minutes * 60
    poll_seconds = 30

    print(f"Waiting for endpoint '{endpoint_name}' to become ready...")
    while time.time() < deadline:
        response = requests.get(status_url, headers=headers)
        state    = response.json().get("state", {})
        ready    = state.get("ready", "")
        config   = state.get("config_update", "")

        print(f"  [{time.strftime('%H:%M:%S')}] Ready={ready}, Config={config}")

        if ready == "READY":
            print("\nEndpoint is ready.")
            return True

        time.sleep(poll_seconds)

    print("\nTimeout waiting for endpoint. Check the Databricks UI for details.")
    return False

wait_for_endpoint(workspace_url, headers, endpoint_name)

### Query the endpoint with sample data

Once the endpoint is ready, any application can call it via HTTP POST with a JSON
payload. The payload format follows the standard MLflow serving schema, where
`dataframe_records` contains a list of row dictionaries. We reconstruct a small
sample from the test set to demonstrate a real end-to-end prediction call.

In [0]:
# Reconstruct the test set from the feature table to get a realistic sample
spark_df    = spark.read.table(f"{catalog}.{schema}.hotel_bookings_features")
bookings_df = spark_df.toPandas()

X = bookings_df.drop(columns=['is_canceled'])
y = bookings_df['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler        = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Take 5 rows as a sample payload
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_train.columns)
sample = X_test_scaled_df.head(5)

# Build the request payload
payload = {"dataframe_records": sample.to_dict(orient="records")}

# Call the endpoint
score_url = f"{workspace_url}/serving-endpoints/{endpoint_name}/invocations"
response  = requests.post(score_url, headers=headers, data=json.dumps(payload))

if response.status_code == 200:
    predictions = response.json()
    print("Predictions from endpoint:")
    print(json.dumps(predictions, indent=2))
    print(f"\nActual labels for these 5 rows: {y_test.iloc[:5].tolist()}")
else:
    print(f"Error querying endpoint: {response.status_code}")
    print(response.json())

## Update the endpoint with a new model version

When a new model version is trained and registered, updating the serving endpoint
requires only a configuration update rather than creating a new endpoint. Traffic
transitions smoothly without downtime.

The pattern below is how you would promote a Challenger to Champion and simultaneously
update the serving endpoint to the new version:

In [0]:
def promote_champion(client, model_uc_path, new_version, workspace_url, headers, endpoint_name):
    """
    Assigns the Champion alias to a new version and updates the serving endpoint.
    Call this after validating a Challenger model's performance.
    """
    # Reassign the Champion alias
    client.set_registered_model_alias(
        name=model_uc_path,
        alias="Champion",
        version=new_version
    )
    print(f"Champion alias reassigned to version {new_version}")

    # Update the serving endpoint to the new version
    update_url    = f"{workspace_url}/api/2.0/serving-endpoints/{endpoint_name}/config"
    update_config = {
        "served_models": [
            {
                "model_name":           model_uc_path,
                "model_version":        str(new_version),
                "workload_size":        "Small",
                "scale_to_zero_enabled": True
            }
        ]
    }

    response = requests.put(update_url, headers=headers, json=update_config)
    if response.status_code == 200:
        print(f"Endpoint '{endpoint_name}' update initiated for version {new_version}.")
    else:
        print(f"Error updating endpoint: {response.status_code}")
        print(response.json())

# Example: promote version 2 when it outperforms the current Champion
# promote_champion(client, model_uc_path, new_version=2,
#                  workspace_url=workspace_url, headers=headers,
#                  endpoint_name=endpoint_name)

print("promote_champion() function defined and ready to use.")

## Summary

This notebook completed the final stage of the ML lifecycle for the hotel
cancellation classifier:

1. **Verified** the model registered in Unity Catalog by the model building notebook,
   confirming version metadata and tracing lineage back to the source training run.

2. **Assigned the Champion alias**, enabling production code to load the model by alias
   rather than by version number, so future version updates require no code changes.

3. **Deployed a Model Serving endpoint**, making the model available as a scalable
   REST API that any application can call with a JSON payload.

4. **Demonstrated endpoint querying**, showing the full round-trip from feature data
   through the REST endpoint to predictions.

5. **Defined a promotion function** that reassigns the Champion alias and updates the
   serving endpoint atomically, providing a repeatable pattern for future model updates.

Together, the four notebooks in this chapter cover the complete Lakehouse ML lifecycle:
feature engineering governed by Unity Catalog, reproducible training tracked by MLflow,
and deployment through a governed registry and a scalable serving endpoint.

## End of notebook